# 16b · GSE232381 · bulk_RNA_seq · the paper's Table 3 genes in the authors' counts

Reads `expression.rds` (notebook 12).

**What the paper reports** (Chen YC et al. *Heliyon* 2024;10:e32303):
- Methods 3.5.2: reads trimmed with FASTP, aligned with HISAT2 2.1.0 and counted with featureCounts 2.0.1;
  "DEGs were identified using DESeq2 (version 1.28.0) or EdgeR (version 3.36.0)". DEGs are differentially
  expressed genes.
- Results 2.4: "Using DESeq2, we identified 28 DEGs" between 9 active and 9 inactive patients (Table 3).
- Methods 3.6: the genes were found with the search of IPA (Ingenuity Pathway Analysis, QIAGEN) for molecules related to glomerulonephritis, lupus
  nephritis, naive and memory T lymphocytes and NK cells, after a log2 fold change cutoff whose value is not
  given.
- Table 3 gives a log2 fold change for each gene. It gives no p values and no significance threshold.

**Which genes can be tested.** Two rows of Table 3 are not single genes ("26s Proteasome complex",
"PI3K (complex)") and are not tested. Where the table names a group and a gene (Jnk | MAPK8, MTORC2 |
RICTOR, MTORC2 | MLST8, Notch | NOTCH4, RAS | RRAS, HSP | DNAJB14), the gene is tested. IGLL1/IGLL5 is
tested as both genes. That leaves 27 genes.

**Samples.** The authors' expected counts for all 25 GEO samples (12 active, 13 inactive; notebook 11).
GEO does not say which 18 the paper compared. The paper chose these genes from its 18 samples, which are
among these 25.

**Two methods, side by side** (active against inactive, all genes, then the paper's genes extracted):
- **DESeq2** (Love MI, Huber W, Anders S. *Genome Biol* 2014;15:550, doi:10.1186/s13059-014-0550-8), the
  method the paper names; default settings, Wald test, Benjamini-Hochberg adjusted p.
- **limma-voom** (Law CW, Chen Y, Shi W, Smyth GK. *Genome Biol* 2014;15:R29, doi:10.1186/gb-2014-15-2-r29),
  after `filterByExpr` and TMM normalisation (Robinson MD, Oshlack A. *Genome Biol* 2010;11:R25).

**Comparison with Table 3.** Direction: the number of genes whose log2 fold change has the paper's sign.
Size: Spearman correlation between our log2 fold changes and the paper's.

## Read the counts

In [1]:
source("../src/paths.R")
suppressMessages({library(DESeq2); library(edgeR); library(limma)})
x <- readRDS(art("GSE232381", "expression.rds"))
counts <- x$all_counts; meta <- x$meta
meta$activity <- factor(ifelse(meta$ln_active == 1, "active", "inactive"), levels = c("inactive", "active"))
c(genes = nrow(counts), samples = ncol(counts)); table(meta$activity)

genes samples 
  25260      25


inactive   active 
      13       12 

**Explanation**
- `library(DESeq2)`, `library(edgeR)` and `library(limma)` provide the two methods (citations in the
  introduction).
- `x$all_counts` holds the authors' expected counts for all genes, named with current NCBI symbols, before
  the low-count filter of notebook 12. Each method applies its own filter.
- `activity` is a factor with inactive first, so the comparison reads active against inactive.

## The paper's genes and log2 fold changes

In [2]:
paper <- data.frame(
  gene = c("CD83", "E2F4", "HBEGF", "HCST", "HLA-G", "DNAJB14", "IGLL1", "IGLL5", "IRAK1", "ITGAV", "MAPK8",
           "MPG", "RICTOR", "MLST8", "NFKBIZ", "NOTCH4", "NR3C1", "PIK3CA", "PKM", "RRAS", "SMAD4", "SP3",
           "TANK", "TBK1", "TGFBR2", "TNFSF12", "ZC3H12A"),
  paper_log2FC = c(-3.14774, -0.49037, -3.57431, -0.67715, -1.55851, 1.327, -3.06243, -3.06243, -0.44746, 1.105709,
                   -2.338, -0.70755, 1.134, -0.630, -4.29614, -1.12, 0.698912, 0.865196, -0.60801, -0.941, 0.722033,
                   0.698179, 0.491583, 0.558364, 0.599042, -0.44374, -2.50818))
c(paper_genes = nrow(paper), in_counts = sum(paper$gene %in% rownames(counts)))

paper_genes   in_counts 
         27          27

**Explanation**
- `paper` holds the 27 testable genes of Table 3 and the log2 fold change the paper reports for each. The
  numbers are copied from Table 3.
- `in_counts` counts how many of them are in the counts table.

**DESeq2**, design `~ activity`, active against inactive.

In [3]:
dds <- DESeqDataSetFromMatrix(round(counts), colData = meta, design = ~ activity)
dds <- suppressMessages(DESeq(dds, quiet = TRUE))
r_deseq <- as.data.frame(results(dds, contrast = c("activity", "active", "inactive")))
c(genes_tested = sum(!is.na(r_deseq$padj)), padj_below_0.05 = sum(r_deseq$padj < 0.05, na.rm = TRUE))

converting counts to integer mode



genes_tested padj_below_0.05 
          12684             107

**Explanation**
- `round(counts)` turns the expected counts into whole numbers: `DESeqDataSetFromMatrix()` accepts only
  whole numbers, and 8.9% of the authors' values are not (notebook 12).
- `DESeqDataSetFromMatrix()` builds DESeq2's object from the counts, the sample table and the design
  `~ activity`.
- `DESeq()` estimates a scaling factor for each sample (size factor) and the variability of each gene
  (dispersion), then fits the model; `quiet = TRUE` hides its messages.
- `results(dds, contrast = c("activity", "active", "inactive"))` gives, for each gene, the log2 fold change of
  active against inactive, its p value and its Benjamini-Hochberg adjusted p value (`padj`).
- DESeq2 sets `padj` to NA for genes it does not test (very low counts or outliers); `!is.na()` counts the
  tested ones.

**Result.** Over all genes, DESeq2 gives 107 genes with adjusted p < 0.05 among 12,684 tested.

**limma-voom**, same design.

In [4]:
design <- model.matrix(~ activity, meta)
dge <- DGEList(counts); keep <- filterByExpr(dge, design)
dge <- calcNormFactors(dge[keep, , keep.lib.sizes = FALSE])
fit <- eBayes(lmFit(voom(dge, design), design))
r_voom <- topTable(fit, coef = "activityactive", number = Inf, sort.by = "none")
c(genes_tested = nrow(r_voom), adj_p_below_0.05 = sum(r_voom$adj.P.Val < 0.05))

genes_tested adj_p_below_0.05 
           12105                0

**Explanation**
- `model.matrix(~ activity, meta)` builds the design matrix: an intercept and a column that is 1 for active
  samples.
- `DGEList(counts)` builds edgeR's object (Robinson MD, McCarthy DJ, Smyth GK. edgeR: a Bioconductor package for
  differential expression analysis of digital gene expression data. *Bioinformatics* 2010;26:139-140);
  `filterByExpr()` keeps genes with enough counts for this design; `calcNormFactors()` computes TMM
  normalization factors. These functions accept counts that are not whole numbers.
- `voom()` turns counts into log2 counts per million with a precision weight for each value; `lmFit()` fits
  the linear model; `eBayes()` moves each gene's variance toward the average over all genes (empirical Bayes).
- `topTable(fit, coef = "activityactive", number = Inf, sort.by = "none")` returns the result for the
  activity column for all genes, in their original order.

**Result.** limma-voom gives no gene with adjusted p < 0.05 among 12,105 tested.

**The paper's genes**, our log2 fold change (active against inactive), p and adjusted p from each method.

In [5]:
cmp <- data.frame(paper,
  DESeq2_log2FC = r_deseq[paper$gene, "log2FoldChange"], DESeq2_p = r_deseq[paper$gene, "pvalue"], DESeq2_padj = r_deseq[paper$gene, "padj"],
  voom_log2FC = r_voom[paper$gene, "logFC"], voom_p = r_voom[paper$gene, "P.Value"], voom_padj = r_voom[paper$gene, "adj.P.Val"])
format(cmp, digits = 3)

,gene,paper_log2FC,DESeq2_log2FC,DESeq2_p,DESeq2_padj,voom_log2FC,voom_p,voom_padj
,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>
1,CD83,-3.148,0.4112,1.31e-01,0.3120,0.156,0.78877,0.881
2,E2F4,-0.490,-0.2782,6.22e-02,0.2057,-0.270,0.08803,0.271
3,HBEGF,-3.574,-0.5601,1.51e-01,0.3384,NA,NA,NA
4,HCST,-0.677,-1.0035,8.28e-04,0.0531,-0.704,0.00977,0.174
5,HLA-G,-1.559,-0.9759,7.83e-02,NA,NA,NA,NA
6,DNAJB14,1.327,0.9883,4.84e-03,0.0759,0.941,0.01700,0.178
7,IGLL1,-3.062,-1.0294,1.18e-01,NA,NA,NA,NA
8,IGLL5,-3.062,0.2996,6.59e-01,0.7995,-0.722,0.49256,0.675
9,IRAK1,-0.447,-0.2818,1.30e-02,0.1006,-0.291,0.03012,0.191


**Explanation**
- One row per paper gene: the paper's log2 fold change next to ours from each method, with p and adjusted p.
  `r_deseq[paper$gene, "log2FoldChange"]` looks up each gene by name; a gene a method did not test is NA.

**Agreement with Table 3**, over the genes each method tested.

In [6]:
agree <- function(ours, padj) {
  ok <- !is.na(ours)
  c(genes = sum(ok), same_direction = sum(sign(ours[ok]) == sign(cmp$paper_log2FC[ok])),
    spearman_rho = round(cor(ours[ok], cmp$paper_log2FC[ok], method = "spearman"), 3),
    spearman_p = signif(cor.test(ours[ok], cmp$paper_log2FC[ok], method = "spearman", exact = FALSE)$p.value, 3),
    adjusted_p_below_0.05 = sum(padj[ok] < 0.05, na.rm = TRUE))
}
rbind(DESeq2 = agree(cmp$DESeq2_log2FC, cmp$DESeq2_padj), limma_voom = agree(cmp$voom_log2FC, cmp$voom_padj))

,genes,same_direction,spearman_rho,spearman_p,adjusted_p_below_0.05
DESeq2,27,23,0.683,8.66e-05,1
limma_voom,24,22,0.779,7.26e-06,0


**Explanation**
- `agree(ours, padj)` compares one method with Table 3 over the genes it tested:
  - `same_direction`: genes whose log2 fold change has the paper's sign;
  - `spearman_rho` and `spearman_p`: Spearman rank correlation of our fold changes with the paper's, and its
    p value. `exact = FALSE` uses the normal approximation, as ties are possible;
  - `adjusted_p_below_0.05`: the paper's genes with adjusted p below 0.05 here.
- `rbind()` puts the rows of the two methods together.

**Result.**
- **Direction.** 23 of 27 genes (DESeq2) and 22 of 24 (limma-voom) have the paper's sign. Our log2 fold
  changes correlate with Table 3: Spearman rho = 0.68 (p = 9e-05) and 0.78 (p = 7e-06).
- **The paper's five largest log2 fold changes** (DESeq2 and limma-voom here):
  - NFKBIZ: paper -4.30; here +0.21 and -0.13;
  - HBEGF: paper -3.57; here -0.56 (not tested by limma-voom);
  - CD83: paper -3.15; here +0.41 and +0.16;
  - IGLL1: paper -3.06; here -1.03 (not tested by limma-voom);
  - ZC3H12A: paper -2.51; here -0.08 and -0.64.
- **Adjusted p.** One of the 27 genes has adjusted p < 0.05 with DESeq2: NOTCH4 (paper -1.12; here -1.18,
  adjusted p = 0.044). None does with limma-voom.
- These are 25 samples (12 active, 13 inactive), not the paper's 18, and the paper selected these genes from
  its 18 samples, which are among these 25.